# exp5 — threshold-free rich interaction table

**Timestamp:** 2026-10-24 01:00 MSK

## Intent / hypothesis
Retain every positive interval overlap between raw view sessions and labelled schedule airings, with no watch threshold in the stored table. This should preserve duration and temporal context for later target thresholds and time-slot features.

**Run ID:** `2026-10-24-exp5-rich-interactions-0100`.


## Method

The builder joins matching `vsetv_id == channel_id` per channel. It uses the reference notebook strict overlap predicate `show_start < view_stop` and `show_stop > view_start`. The reference implementation is `ndmel/2nd_place_recsys_cinema_challenge_2020/1_prepare_data.ipynb`, cells 2, 4, and 8 (source available through the GitHub API). It does not threshold watch duration. The local builder additionally rejects schedule interval overlap before using binary-search bounds.


In [1]:
! .venv/bin/python artifacts/scripts/exp5_build_rich_interactions.py

[exp5] output=/Users/davidkk/Kaggle_Case/data/exp5_user_airings_rich.parquet
[exp5] rows=3,847,696; users=4,895; shows=4,363
[exp5] overlap seconds min/median/max=1.000000/523.000000/30000.000000
[exp5] watch_pct min/median/max=0.000034/0.116087/1.000000
[exp5] target counts >=.3/.5/.8=1,238,985/885,038/570,588
[exp5] elapsed_s=10.7; parquet_bytes=77,905,975


In [2]:
import pandas as pd
d = pd.read_parquet("data/exp5_user_airings_rich.parquet")
print("shape:", d.shape)
print("null cells:", int(d.isna().sum().sum()))
print("interval predicate violations:", int((~((d.view_start < d.show_stop) & (d.view_stop > d.show_start))).sum()))
print("threshold counts:", {str(t): int(d.user_watch_pct.ge(t).sum()) for t in (.3, .5, .8)})
print("weeks:", int(d.week.min()), int(d.week.max()))

shape: (3847696, 14)
null cells: 0
interval predicate violations: 0
threshold counts: {'0.3': 1238985, '0.5': 885038, '0.8': 570588}
weeks: 11 30


## Result interpretation

The new parquet contains all 14 required columns, 3,847,696 rows, and no null cells. Every stored row satisfies strict interval overlap; minimum overlap is one second. Thresholds are measurable downstream, not baked into the interaction table. These are data-construction checks only, not MAP@5 results; target construction, candidate recall, and ranking experiments remain pending.

The sampled reference cross-check is an exact multiset match, with zero missing or extra pairs.


## Independent reference cross-check

A naïve implementation of the exact overlap predicate from reference notebook cell 8 was run against 5,000 deterministic raw-view samples. The 3,319 expected pairs were compared as a multiset to the constructed parquet.

Run ID: `2026-10-24-exp5-reference-check-0125`.


In [3]:
! .venv/bin/python artifacts/scripts/exp5_reference_check.py

sample views: 5000 reference pairs: 3319 rich-table pairs: 3319
reference-only: 0 table-only: 0 exact multiset match: True
